In [11]:
import os
import sys
from pathlib import Path
import builtins

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import spearmanr

from sklearn.linear_model import Ridge, ElasticNet, SGDRegressor, Lasso, HuberRegressor
from sklearn.svm import LinearSVR, SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

import joblib

SEED = 42
np.random.seed(SEED)

TARGET_COL = "target"
TIME_COL   = "timestamp"

WINDOW  = 6
HORIZON = 1

EPS = 1e-8


In [12]:
# Change working directory to project root
PROJECT_ROOT = Path().absolute().parent if Path().absolute().name == 'infer' else Path().absolute()
os.chdir(PROJECT_ROOT)
!pwd

/Users/phatvu/Documents/Dev-Drive-Local/crypto-price-forecaster-glm


In [13]:
# Import project configuration
import sys
sys.path.append('.')
from config import *

In [14]:
PROJECT_ROOT = Path().absolute()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "final.pkl"

assert DATA_PATH.exists(), f"Not found: {DATA_PATH}"
master_df_final = pd.read_pickle(DATA_PATH)

master_df_final[ TIME_COL ] = pd.to_datetime(master_df_final[ TIME_COL ])
master_df_final = master_df_final.sort_values(TIME_COL).reset_index(drop=True)

print("Loaded:", master_df_final.shape)
master_df_final.head()


Loaded: (16328, 32)


,timestamp,BTC_log_return,ETH_log_return,target,BTC_vol_log_return,log_rvol,vol_gk,vol_gk_z,vol_signed,clv,...,log_puell,onchain_mvrv_z,onchain_netflow_z,reserve_ratio_change,news_article_count,news_impact_score,news_sentiment_decay_slow,news_sentiment_shock,fng_change,fng_divergence
0,2018-06-02 16:00:00,0.005481,0.012730,-0.003263,-0.050970,-0.082714,0.006501,-1.055547,0.006501,0.823299,...,0.048605,-0.935453,2.000939,0.000000,0.0,0.0,0.032102,0.0,0.0,2.666667
1,2018-06-02 20:00:00,-0.003263,-0.012798,0.000402,-0.312065,-0.393486,0.005600,-1.188620,-0.005600,0.006371,...,0.047732,-0.933289,1.817131,0.000000,0.0,0.0,0.022930,0.0,0.0,2.571429
2,2018-06-03 00:00:00,0.000402,0.004745,0.010011,0.095915,-0.249293,0.007028,-0.876730,0.007028,0.132422,...,0.025592,-0.876055,0.545649,0.000049,0.0,0.0,0.016378,0.0,13.0,15.071429
3,2018-06-03 04:00:00,0.010011,0.022107,-0.000667,0.221492,-0.010322,0.013549,0.387265,0.013549,0.175088,...,0.024780,-0.874065,0.503094,0.000000,0.0,0.0,0.011699,0.0,0.0,14.571429
4,2018-06-03 08:00:00,-0.000667,0.021193,0.001963,0.099927,0.084544,0.007369,-0.781489,-0.007369,-0.520743,...,0.023969,-0.872080,0.460896,0.000000,0.0,0.0,0.008356,0.0,0.0,14.071429


In [15]:
X = master_df_final.drop(columns=[TARGET_COL, TIME_COL])
y = master_df_final[TARGET_COL].copy()
timestamps = master_df_final[TIME_COL].copy()

n = len(master_df_final)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

X_train = X.iloc[:train_end].copy()
y_train = y.iloc[:train_end].copy()

X_val   = X.iloc[train_end:val_end].copy()
y_val   = y.iloc[train_end:val_end].copy()

X_test  = X.iloc[val_end:].copy()
y_test  = y.iloc[val_end:].copy()

print("Shapes:", X_train.shape, X_val.shape, X_test.shape)

# no leakage sanity
assert X_train.index.max() < X_val.index.min()
assert X_val.index.max() < X_test.index.min()


Shapes: (11429, 30) (2449, 30) (2450, 30)


In [16]:
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_val_scaled = pd.DataFrame(
    scaler.transform(X_val),
    columns=X_val.columns,
    index=X_val.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

display(X_train_scaled.describe().loc[["mean", "std"]].T.head(10))

# no leakage sanity
assert X_train_scaled.index.max() < X_val_scaled.index.min()
assert X_val_scaled.index.max() < X_test_scaled.index.min()


,mean,std
BTC_log_return,1.865105e-18,1.000044
ETH_log_return,4.351911e-18,1.000044
BTC_vol_log_return,-3.302790e-18,1.000044
log_rvol,-1.740764e-17,1.000044
vol_gk,-6.963058e-17,1.000044
vol_gk_z,-5.595314e-18,1.000044
vol_signed,1.181233e-17,1.000044
clv,-3.730209e-17,1.000044
wick_imbalance,-2.486806e-17,1.000044
bb_pct_b,2.586279e-16,1.000044


In [17]:
def make_sliding_window_xy(
    X: pd.DataFrame,
    y: pd.Series,
    window: int = 6,
    horizon: int = 1,
    dropna: bool = True
):
    """
    Predict y at t+horizon using features from X[t-window+1 ... t].
    Output index aligns to target timestamp (t+horizon).
    """
    X = X.copy()
    y = y.copy()

    common_idx = X.index.intersection(y.index)
    X = X.loc[common_idx]
    y = y.loc[common_idx]

    if dropna:
        mask = X.notna().all(axis=1) & y.notna()
        X = X.loc[mask]
        y = y.loc[mask]

    n = len(X)
    if n < window + horizon:
        raise ValueError(f"Not enough rows: have {n}, need >= {window + horizon}")

    X_vals = X.values
    y_vals = y.values

    rows, targets, out_index = [], [], []

    for t in builtins.range(window - 1, n - horizon):
        x_block = X_vals[t - window + 1 : t + 1]   # (window, n_features)
        rows.append(x_block.reshape(-1))           # flatten to 1D
        targets.append(y_vals[t + horizon])        # future target
        out_index.append(X.index[t + horizon])     # align to target row index

    feat_names = list(X.columns)
    cols = []
    for lag in builtins.range(window, 0, -1):      # lag{window} oldest ... lag1 newest
        for f in feat_names:
            cols.append(f"{f}_lag{lag}")

    Xw = pd.DataFrame(rows, columns=cols, index=out_index)
    yw = pd.Series(targets, index=out_index, name=y.name if y.name else "target")
    return Xw, yw


Xtr_w, ytr_w = make_sliding_window_xy(X_train_scaled, y_train, window=WINDOW, horizon=HORIZON)
Xva_w, yva_w = make_sliding_window_xy(X_val_scaled,   y_val,   window=WINDOW, horizon=HORIZON)
Xte_w, yte_w = make_sliding_window_xy(X_test_scaled,  y_test,  window=WINDOW, horizon=HORIZON)

print("Windowed shapes:")
print("Train:", Xtr_w.shape, ytr_w.shape)
print("Val  :", Xva_w.shape, yva_w.shape)
print("Test :", Xte_w.shape, yte_w.shape)

# no leakage (windowed index still respects ordering)
assert Xtr_w.index.max() < Xva_w.index.min()
assert Xva_w.index.max() < Xte_w.index.min()


Windowed shapes:
Train: (11423, 180) (11423,)
Val  : (2443, 180) (2443,)
Test : (2444, 180) (2444,)


In [18]:
def eval_regression(
    y_true,
    y_pred,
    capital: float = 1_000_000_000,
    threshold: float = 0.0,
    fee_bps: float = 0.0
):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)

    ic = spearmanr(y_true, y_pred).correlation
    dir_acc = (np.sign(y_true) == np.sign(y_pred)).mean()

    # position rule
    pos = np.where(y_pred > threshold, 1,
          np.where(y_pred < -threshold, -1, 0))

    gross_pnl = capital * pos * y_true
    fee = (fee_bps / 10_000) * capital
    net_pnl = gross_pnl - (np.abs(pos) * fee)

    total_pnl = net_pnl.sum()
    avg_pnl   = net_pnl.mean()
    trades    = int((pos != 0).sum())
    win_rate  = (net_pnl[pos != 0] > 0).mean() if trades > 0 else np.nan

    equity = np.cumsum(net_pnl)
    peak = np.maximum.accumulate(equity)
    drawdown = equity - peak
    max_dd = drawdown.min()

    return {
        "RMSE": rmse,
        "MAE": mae,
        "IC_spearman": ic,
        "Directional_Acc": dir_acc,
        "PnL_1B_Total": total_pnl,
        "PnL_1B_AvgPerBar": avg_pnl,
        "Trades": trades,
        "WinRate": win_rate,
        "MaxDrawdown_1B": max_dd
    }


In [19]:
# %%
pred0_val  = np.zeros(len(yva_w))
pred0_test = np.zeros(len(yte_w))

pred_last_val  = yva_w.shift(1).fillna(0).values
pred_last_test = yte_w.shift(1).fillna(0).values

baseline_rows = [
    {"Model": "BASELINE_zero",
     **{f"VAL_{k}": v for k, v in eval_regression(yva_w, pred0_val).items()},
     **{f"TEST_{k}": v for k, v in eval_regression(yte_w, pred0_test).items()}},

    {"Model": "BASELINE_last",
     **{f"VAL_{k}": v for k, v in eval_regression(yva_w, pred_last_val).items()},
     **{f"TEST_{k}": v for k, v in eval_regression(yte_w, pred_last_test).items()}},
]


/var/folders/h3/4_pq_7pd3pq2286xkp78c1480000gn/T/ipykernel_23876/268402780.py:15: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic = spearmanr(y_true, y_pred).correlation
/var/folders/h3/4_pq_7pd3pq2286xkp78c1480000gn/T/ipykernel_23876/268402780.py:15: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic = spearmanr(y_true, y_pred).correlation


In [20]:
models = {
    # --- Linear / Regularization ---
    "Ridge": Ridge(alpha=1.0, random_state=SEED),
    "Lasso": Lasso(alpha=1e-4, max_iter=20000, random_state=SEED),
    "ElasticNet": ElasticNet(alpha=1e-3, l1_ratio=0.2, random_state=SEED, max_iter=20000),

    # Robust to outliers
    "Huber": HuberRegressor(epsilon=1.35, alpha=1e-4, max_iter=1000),

    # Large-scale linear
    "SGDRegressor": SGDRegressor(
        loss="squared_error",
        penalty="l2",
        alpha=1e-4,
        learning_rate="optimal",
        max_iter=10000,
        tol=1e-4,
        random_state=SEED
    ),

    # --- SVM ---
    "LinearSVR": LinearSVR(C=1.0, epsilon=5e-4, random_state=SEED, max_iter=10000),
    "SVR_RBF": SVR(kernel="rbf", C=1.0, gamma="scale", epsilon=5e-4),

    # --- Decision Tree ---
    "DecisionTree": DecisionTreeRegressor(
        max_depth=5,
        min_samples_leaf=50,
        random_state=SEED
    ),

    # --- Ensemble (Bagging) ---
    "RandomForest": RandomForestRegressor(
        n_estimators=300,
        max_depth=6,
        min_samples_leaf=30,
        n_jobs=-1,
        random_state=SEED
    ),

    # --- Ensemble (Boosting) ---
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=SEED
    ),
}

results = []
fitted_models = {}

from tqdm import tqdm
for name, model in tqdm(models.items()):
    model.fit(Xtr_w, ytr_w)
    fitted_models[name] = model

    pred_val  = model.predict(Xva_w)
    pred_test = model.predict(Xte_w)

    results.append({
        "Model": name,
        **{f"VAL_{k}": v for k, v in eval_regression(yva_w, pred_val).items()},
        **{f"TEST_{k}": v for k, v in eval_regression(yte_w, pred_test).items()},
    })

df_results = pd.DataFrame(baseline_rows + results).sort_values(by="VAL_IC_spearman", ascending=False)
display(df_results)

best_model_name = df_results.iloc[0]["Model"]
print("Best by VAL_IC_spearman:", best_model_name)


 30%|███       | 3/10 [00:01<00:02,  2.58it/s]/Users/phatvu/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_huber.py:348: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
 50%|█████     | 5/10 [00:08<00:10,  2.00s/it]/Users/phatvu/miniconda3/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
100%|██████████| 10/10 [06:36<00:00, 39.68s/it]


,Model,VAL_RMSE,VAL_MAE,VAL_IC_spearman,VAL_Directional_Acc,VAL_PnL_1B_Total,VAL_PnL_1B_AvgPerBar,VAL_Trades,VAL_WinRate,VAL_MaxDrawdown_1B,TEST_RMSE,TEST_MAE,TEST_IC_spearman,TEST_Directional_Acc,TEST_PnL_1B_Total,TEST_PnL_1B_AvgPerBar,TEST_Trades,TEST_WinRate,TEST_MaxDrawdown_1B
3,Lasso,9.449931e-03,6.471568e-03,0.298918,0.578387,5.690483e+09,2.329301e+06,2443,0.578387,-1.099508e+08,9.368898e-03,6.459390e-03,0.269264,0.578560,5.281123e+09,2.160852e+06,2444,0.578560,-9.247718e+07
5,Huber,9.494275e-03,6.531933e-03,0.294941,0.574703,5.689762e+09,2.329006e+06,2443,0.574703,-1.370570e+08,1.574007e-02,7.684320e-03,0.263710,0.587561,5.099126e+09,2.086386e+06,2444,0.587561,-1.296407e+08
2,Ridge,9.765033e-03,6.832251e-03,0.282656,0.566107,5.358628e+09,2.193462e+06,2443,0.566107,-1.357353e+08,1.583633e-02,8.130871e-03,0.245012,0.574468,4.767449e+09,1.950675e+06,2444,0.574468,-1.372277e+08
4,ElasticNet,9.611308e-03,6.478202e-03,0.276315,0.566926,5.339202e+09,2.185510e+06,2443,0.566926,-1.099508e+08,9.131867e-03,6.220373e-03,0.245666,0.562602,4.898540e+09,2.004313e+06,2444,0.562602,-7.552525e+07
7,LinearSVR,1.288905e-02,8.854532e-03,0.180510,0.559558,3.875537e+09,1.586384e+06,2443,0.559558,-1.918956e+08,3.706577e-02,1.191604e-02,0.147933,0.540507,3.357580e+09,1.373805e+06,2444,0.540507,-1.064592e+08
8,SVR_RBF,1.187485e-02,8.843089e-03,0.111883,0.518215,2.924389e+09,1.197049e+06,2443,0.518215,-2.496318e+08,1.138708e-02,8.518916e-03,0.141409,0.531506,2.799381e+09,1.145409e+06,2444,0.531506,-2.507479e+08
11,GradientBoosting,1.026868e-02,6.849245e-03,0.110189,0.526402,2.867113e+09,1.173603e+06,2443,0.526402,-1.686690e+08,9.656183e-03,6.598872e-03,0.097390,0.527414,1.986237e+09,8.126993e+05,2444,0.527414,-2.293151e+08
10,RandomForest,1.016011e-02,6.729737e-03,0.100981,0.526811,2.635208e+09,1.078677e+06,2443,0.526811,-2.633463e+08,9.525166e-03,6.392917e-03,0.090277,0.518003,2.475067e+09,1.012711e+06,2444,0.518003,-1.614355e+08
9,DecisionTree,1.035889e-02,6.892992e-03,0.056616,0.520262,9.596124e+08,3.928008e+05,2443,0.520262,-2.621861e+08,9.690862e-03,6.470370e-03,0.042414,0.508183,8.267461e+08,3.382758e+05,2444,0.508183,-3.245856e+08
6,SGDRegressor,1.179533e+11,3.552240e+10,0.013851,0.512894,1.228261e+08,5.027676e+04,2443,0.512894,-4.575997e+08,1.321732e+11,4.340317e+10,-0.006167,0.486907,3.104481e+08,1.270246e+05,2444,0.486907,-4.871739e+08


Best by VAL_IC_spearman: Lasso


In [21]:
# %%
OUT_DIR = PROJECT_ROOT / "artifacts"
OUT_DIR.mkdir(parents=True, exist_ok=True)

if best_model_name in fitted_models:
    best_model = fitted_models[best_model_name]

    pred_val  = best_model.predict(Xva_w)
    pred_test = best_model.predict(Xte_w)

    print("VAL:",  eval_regression(yva_w, pred_val))
    print("TEST:", eval_regression(yte_w, pred_test))

    joblib.dump(scaler, OUT_DIR / "scaler.joblib")
    joblib.dump(best_model, OUT_DIR / f"model_{best_model_name}.joblib")

    meta = {
        "best_model": best_model_name,
        "window": int(WINDOW),
        "horizon": int(HORIZON),
        "target_col": TARGET_COL,
        "time_col": TIME_COL,
        "n_rows": int(n),
        "train_end": int(train_end),
        "val_end": int(val_end),
        "feature_cols": list(X.columns),
        "windowed_feature_cols": list(Xtr_w.columns),
    }
    pd.Series(meta).to_json(OUT_DIR / "train_meta.json")

    print("Saved to:", OUT_DIR)
else:
    print("Best model is a baseline; nothing to save.")


VAL: {'RMSE': np.float64(0.00944993075251409), 'MAE': 0.006471568343003941, 'IC_spearman': np.float64(0.2989180231757809), 'Directional_Acc': np.float64(0.5783872288170282), 'PnL_1B_Total': np.float64(5690483198.97102), 'PnL_1B_AvgPerBar': np.float64(2329301.3503770037), 'Trades': 2443, 'WinRate': np.float64(0.5783872288170282), 'MaxDrawdown_1B': np.float64(-109950817.74223089)}
TEST: {'RMSE': np.float64(0.00936889820634891), 'MAE': 0.006459390233377578, 'IC_spearman': np.float64(0.2692643767460621), 'Directional_Acc': np.float64(0.5785597381342062), 'PnL_1B_Total': np.float64(5281122772.771433), 'PnL_1B_AvgPerBar': np.float64(2160852.19835165), 'Trades': 2444, 'WinRate': np.float64(0.5785597381342062), 'MaxDrawdown_1B': np.float64(-92477181.2920618)}
Saved to: /Users/phatvu/Documents/Dev-Drive-Local/crypto-price-forecaster-glm/artifacts
